In [0]:
spark.sql("DROP TABLE IF EXISTS default.usaspending_state_quarter_silver")
spark.sql("DROP TABLE IF EXISTS default.usaspending_state_quarter_gold")
spark.sql("DROP TABLE IF EXISTS default.usaspending_state_year_gold")

DataFrame[]

In [0]:
# =========================================================
# USAspending Enterprise Medallion Config
# =========================================================

country = "USA"
country_code = "US"
geo_level = "state"

states = ["PA", "NJ", "NY", "CA", "TX", "FL"]
years = [2024, 2025, 2026]

quarters = {
    "Q1": ("01-01", "03-31"),
    "Q2": ("04-01", "06-30"),
    "Q3": ("07-01", "09-30"),
    "Q4": ("10-01", "12-31"),
}

pipeline_name = "usaspending_medallion_databricks"
source_system = "USAspending API"
refresh_mode = "FULL"
environment = "DEV"

In [0]:
print(states)
print(years)
print (quarters)

['PA', 'NJ', 'NY', 'CA', 'TX', 'FL']
[2024, 2025, 2026]
{'Q1': ('01-01', '03-31'), 'Q2': ('04-01', '06-30'), 'Q3': ('07-01', '09-30'), 'Q4': ('10-01', '12-31')}


In [0]:
import requests
import pandas as pd
from pyspark.sql import functions as F

endpoint = "https://api.usaspending.gov/api/v2/search/spending_over_time/"

rows = []

for year in years:
    for quarter, dates in quarters.items():
        for state in states:
              body = {
                "group": "quarter",
                "subawards": False,
                "filters": {
                    "time_period": [{
                        "start_date": f"{year}-{dates[0]}",
                        "end_date": f"{year}-{dates[1]}"
                    }],
                    "place_of_performance_scope": "domestic",
                    "place_of_performance_locations": [
                        {"country": "USA", "state": state}
                    ],
                    "award_type_codes": ["A", "B", "C", "D", "02", "03", "04", "05"]
                }
            }

import time

rows = []

quarters = {
    "Q1": ("01-01", "03-31"),
    "Q2": ("04-01", "06-30"),
    "Q3": ("07-01", "09-30"),
    "Q4": ("10-01", "12-31"),
}

for year in years:
    for quarter, dates in quarters.items():
        for state in states:

            body = {
                "group": "quarter",
                "subawards": False,
                "filters": {
                    "time_period": [{
                        "start_date": f"{year}-{dates[0]}",
                        "end_date": f"{year}-{dates[1]}"
                    }],
                    "place_of_performance_scope": "domestic",
                    "place_of_performance_locations": [
                        {"country": "USA", "state": state}
                    ],
                    "award_type_codes": ["A", "B", "C", "D", "02", "03", "04", "05"]
                }
            }

            success = False

            for attempt in range(3):

                try:
                    r = requests.post(endpoint, json=body, timeout=120)

                    if r.status_code == 200:
                        payload = r.json()
                        success = True
                        time.sleep(1)
                        break

                    print(f"Retry {attempt+1}: HTTP {r.status_code}")

                except Exception as e:
                    print(f"Retry {attempt+1} failed: {e}")

                time.sleep(5)

            if not success:
                print(f"Skipping {state} {year} {quarter}")
                continue

            total = sum(
                float(x.get("aggregated_amount", 0) or 0)
                for x in payload.get("results", [])
            )

            count = sum(
                int(x.get("transaction_count", 0) or 0)
                for x in payload.get("results", [])
            )

            rows.append({
                "country": country,
                "country_code": country_code,
                "geo_level": geo_level,
                "state": state,
                "year": year,
                "quarter": quarter,
                "period": f"{year}-{quarter}",
                "total_obligations": total,
                "transaction_count": count,
                "source_system": source_system
            })

silver_df = spark.createDataFrame(pd.DataFrame(rows))

silver_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("default.usaspending_state_quarter_silver")

display(silver_df)

Retry 1 failed: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
Retry 2 failed: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))


country,country_code,geo_level,state,year,quarter,period,total_obligations,transaction_count,source_system
USA,US,state,PA,2024,Q1,2024-Q1,1.285273258249E10,0,USAspending API
USA,US,state,NJ,2024,Q1,2024-Q1,7.42079132362E9,0,USAspending API
USA,US,state,NY,2024,Q1,2024-Q1,2.80219087863E10,0,USAspending API
USA,US,state,CA,2024,Q1,2024-Q1,5.266828891857E10,0,USAspending API
USA,US,state,TX,2024,Q1,2024-Q1,2.570212140284E10,0,USAspending API
USA,US,state,FL,2024,Q1,2024-Q1,1.7965954629370003E10,0,USAspending API
USA,US,state,PA,2024,Q2,2024-Q2,1.861540130384E10,0,USAspending API
USA,US,state,NJ,2024,Q2,2024-Q2,7.87020199433E9,0,USAspending API
USA,US,state,NY,2024,Q2,2024-Q2,2.973427289062E10,0,USAspending API
USA,US,state,CA,2024,Q2,2024-Q2,5.621595714562E10,0,USAspending API


In [0]:
display(silver_df.select("country", "country_code", "geo_level", "state", "year", "quarter").limit(20))

country,country_code,geo_level,state,year,quarter
USA,US,state,PA,2024,Q1
USA,US,state,NJ,2024,Q1
USA,US,state,NY,2024,Q1
USA,US,state,CA,2024,Q1
USA,US,state,TX,2024,Q1
USA,US,state,FL,2024,Q1
USA,US,state,PA,2024,Q2
USA,US,state,NJ,2024,Q2
USA,US,state,NY,2024,Q2
USA,US,state,CA,2024,Q2


In [0]:
expected_states = ["PA","NJ","NY","CA","TX","FL"]

invalid_states = silver_df.filter(
    ~F.col("state").isin(expected_states)
)

display(invalid_states)

country,country_code,geo_level,state,year,quarter,period,total_obligations,transaction_count,source_system


In [0]:
gold_quarter = (
    silver_df
    .groupBy("country", "country_code", "geo_level", "state", "year", "quarter", "period")
    .agg(
        F.sum("total_obligations").alias("total_obligations"),
        F.sum("transaction_count").alias("transaction_count")
    )
)

gold_year = (
    silver_df
    .groupBy("country", "country_code", "geo_level", "state", "year")
    .agg(
        F.sum("total_obligations").alias("total_obligations"),
        F.sum("transaction_count").alias("transaction_count"),
        F.countDistinct("quarter").alias("quarters_reported")
    )
)

gold_quarter.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "default.usaspending_state_quarter_gold"
)

gold_year.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "default.usaspending_state_year_gold"
)

display(gold_quarter)

country,country_code,geo_level,state,year,quarter,period,total_obligations,transaction_count
USA,US,state,PA,2024,Q1,2024-Q1,1.285273258249E10,0
USA,US,state,NJ,2024,Q1,2024-Q1,7.42079132362E9,0
USA,US,state,NY,2024,Q1,2024-Q1,2.80219087863E10,0
USA,US,state,CA,2024,Q1,2024-Q1,5.266828891857E10,0
USA,US,state,TX,2024,Q1,2024-Q1,2.570212140284E10,0
USA,US,state,FL,2024,Q1,2024-Q1,1.7965954629370003E10,0
USA,US,state,PA,2024,Q2,2024-Q2,1.861540130384E10,0
USA,US,state,NJ,2024,Q2,2024-Q2,7.87020199433E9,0
USA,US,state,NY,2024,Q2,2024-Q2,2.973427289062E10,0
USA,US,state,CA,2024,Q2,2024-Q2,5.621595714562E10,0


In [0]:
%sql
SELECT DISTINCT country, country_code, geo_level, state
FROM default.usaspending_state_quarter_gold
ORDER BY country, geo_level, state;

country,country_code,geo_level,state
USA,US,state,CA
USA,US,state,FL
USA,US,state,NJ
USA,US,state,NY
USA,US,state,PA
USA,US,state,TX


In [0]:
from pyspark.sql import functions as F

gold_quarter = (
    silver_df
    .groupBy("country", "country_code", "geo_level", "state", "year", "quarter", "period")
    .agg(
        F.sum("total_obligations").alias("total_obligations"),
        F.sum("transaction_count").alias("transaction_count")
    )
)

gold_year = (
    silver_df
    .groupBy("country", "country_code", "geo_level", "state", "year")
    .agg(
        F.sum("total_obligations").alias("total_obligations"),
        F.sum("transaction_count").alias("transaction_count"),
        F.countDistinct("quarter").alias("quarters_reported")
    )
)

gold_quarter.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("default.usaspending_state_quarter_gold")

gold_year.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("default.usaspending_state_year_gold")

display(gold_quarter)

country,country_code,geo_level,state,year,quarter,period,total_obligations,transaction_count
USA,US,state,PA,2024,Q1,2024-Q1,1.285273258249E10,0
USA,US,state,NJ,2024,Q1,2024-Q1,7.42079132362E9,0
USA,US,state,NY,2024,Q1,2024-Q1,2.80219087863E10,0
USA,US,state,CA,2024,Q1,2024-Q1,5.266828891857E10,0
USA,US,state,TX,2024,Q1,2024-Q1,2.570212140284E10,0
USA,US,state,FL,2024,Q1,2024-Q1,1.7965954629370003E10,0
USA,US,state,PA,2024,Q2,2024-Q2,1.861540130384E10,0
USA,US,state,NJ,2024,Q2,2024-Q2,7.87020199433E9,0
USA,US,state,NY,2024,Q2,2024-Q2,2.973427289062E10,0
USA,US,state,CA,2024,Q2,2024-Q2,5.621595714562E10,0


In [0]:
%sql
SELECT country, country_code, geo_level, state, year, quarter, period
FROM default.usaspending_state_quarter_gold
ORDER BY year, quarter, state;

country,country_code,geo_level,state,year,quarter,period
USA,US,state,CA,2024,Q1,2024-Q1
USA,US,state,FL,2024,Q1,2024-Q1
USA,US,state,NJ,2024,Q1,2024-Q1
USA,US,state,NY,2024,Q1,2024-Q1
USA,US,state,PA,2024,Q1,2024-Q1
USA,US,state,TX,2024,Q1,2024-Q1
USA,US,state,CA,2024,Q2,2024-Q2
USA,US,state,FL,2024,Q2,2024-Q2
USA,US,state,NJ,2024,Q2,2024-Q2
USA,US,state,NY,2024,Q2,2024-Q2


In [0]:
from pyspark.sql import functions as F
from datetime import datetime

from datetime import datetime, timezone

refresh_ts = datetime.now(timezone.utc).isoformat()

observability_refresh = spark.createDataFrame([
    {
        "pipeline_name": "usaspending_medallion_databricks",
        "refresh_timestamp_utc": refresh_ts,
        "source": "USAspending API",
        "states_requested": ",".join(states),
        "years_requested": ",".join([str(y) for y in years]),
        "row_count_silver": silver_df.count(),
        "row_count_gold_quarter": gold_quarter.count(),
        "row_count_gold_year": gold_year.count(),
        "status": "SUCCESS"
    }
])

freshness = (
    silver_df
    .groupBy("state")
    .agg(
        F.max("year").alias("latest_year"),
        F.max("period").alias("latest_period"),
        F.count("*").alias("period_count"),
        F.sum("total_obligations").alias("total_obligations")
    )
    .withColumn("refresh_timestamp_utc", F.lit(refresh_ts))
)

quality = spark.createDataFrame([
    {
        "metric_name": "silver_null_state_count",
        "metric_value": silver_df.filter(F.col("state").isNull()).count(),
        "refresh_timestamp_utc": refresh_ts
    },
    {
        "metric_name": "silver_null_period_count",
        "metric_value": silver_df.filter(F.col("period").isNull()).count(),
        "refresh_timestamp_utc": refresh_ts
    },
    {
        "metric_name": "silver_negative_obligations_count",
        "metric_value": silver_df.filter(F.col("total_obligations") < 0).count(),
        "refresh_timestamp_utc": refresh_ts
    },
    {
        "metric_name": "gold_quarter_row_count",
        "metric_value": gold_quarter.count(),
        "refresh_timestamp_utc": refresh_ts
    }
])

observability_refresh.write.format("delta").mode("append").saveAsTable(
    "default.usaspending_observability_refresh_log"
)

freshness.write.format("delta").mode("overwrite").saveAsTable(
    "default.usaspending_observability_freshness"
)

quality.write.format("delta").mode("append").saveAsTable(
    "default.usaspending_observability_quality"
)

display(observability_refresh)

pipeline_name,refresh_timestamp_utc,row_count_gold_quarter,row_count_gold_year,row_count_silver,source,states_requested,status,years_requested
usaspending_medallion_databricks,2026-05-25T21:44:22.598755+00:00,72,18,72,USAspending API,"PA,NJ,NY,CA,TX,FL",SUCCESS,"2024,2025,2026"


In [0]:
%sql
SELECT *
FROM default.usaspending_observability_refresh_log
ORDER BY refresh_timestamp_utc DESC;

pipeline_name,refresh_timestamp_utc,row_count_gold_quarter,row_count_gold_year,row_count_silver,source,states_requested,status,years_requested
usaspending_medallion_databricks,2026-05-25T21:44:22.598755+00:00,72,18,72,USAspending API,"PA,NJ,NY,CA,TX,FL",SUCCESS,"2024,2025,2026"
usaspending_medallion_databricks,2026-05-25T21:43:37.221502+00:00,72,18,72,USAspending API,"PA,NJ,NY,CA,TX,FL",SUCCESS,"2024,2025,2026"
usaspending_medallion_databricks,2026-05-25T21:25:41.962954+00:00,72,18,72,USAspending API,"PA,NJ,NY,CA,TX,FL",SUCCESS,"2024,2025,2026"
usaspending_medallion_databricks,2026-05-25T20:57:16.267484+00:00,72,18,72,USAspending API,"PA,NJ,NY,CA,TX,FL",SUCCESS,"2024,2025,2026"
usaspending_medallion_databricks,2026-05-25T20:36:25.563543+00:00,72,18,72,USAspending API,"PA,NJ,NY,CA,TX,FL",SUCCESS,"2024,2025,2026"
usaspending_medallion_databricks,2026-05-25T16:08:52.461003+00:00,72,18,72,USAspending API,"PA,NJ,NY,CA,TX,FL",SUCCESS,"2024,2025,2026"
usaspending_medallion_databricks,2026-05-25T15:40:35.903790+00:00,72,18,72,USAspending API,"PA,NJ,NY,CA,TX,FL",SUCCESS,"2024,2025,2026"
usaspending_medallion_databricks,2026-05-25T15:34:12.366572+00:00,60,15,60,USAspending API,"PA,NJ,NY,CA,TX, FL",SUCCESS,"2024,2025,2026"
usaspending_medallion_databricks,2026-05-25T15:26:28.757543+00:00,40,10,40,USAspending API,"PA,NJ,NY,CA,TX",SUCCESS,"2024,2025"
usaspending_medallion_databricks,2026-05-25T15:10:55.374480+00:00,40,10,40,USAspending API,"PA,NJ,NY,CA,TX",SUCCESS,"2024,2025"


In [0]:
%sql
SELECT *
FROM default.usaspending_observability_quality
ORDER BY refresh_timestamp_utc DESC, metric_name;

metric_name,metric_value,refresh_timestamp_utc
gold_quarter_row_count,72,2026-05-25T21:44:22.598755+00:00
silver_negative_obligations_count,0,2026-05-25T21:44:22.598755+00:00
silver_null_period_count,0,2026-05-25T21:44:22.598755+00:00
silver_null_state_count,0,2026-05-25T21:44:22.598755+00:00
gold_quarter_row_count,72,2026-05-25T21:43:37.221502+00:00
silver_negative_obligations_count,0,2026-05-25T21:43:37.221502+00:00
silver_null_period_count,0,2026-05-25T21:43:37.221502+00:00
silver_null_state_count,0,2026-05-25T21:43:37.221502+00:00
gold_quarter_row_count,72,2026-05-25T21:25:41.962954+00:00
silver_negative_obligations_count,0,2026-05-25T21:25:41.962954+00:00


In [0]:
%sql
SELECT country, country_code, geo_level, state, year, quarter, period
FROM default.usaspending_state_quarter_gold
ORDER BY year, quarter, state;

country,country_code,geo_level,state,year,quarter,period
USA,US,state,CA,2024,Q1,2024-Q1
USA,US,state,FL,2024,Q1,2024-Q1
USA,US,state,NJ,2024,Q1,2024-Q1
USA,US,state,NY,2024,Q1,2024-Q1
USA,US,state,PA,2024,Q1,2024-Q1
USA,US,state,TX,2024,Q1,2024-Q1
USA,US,state,CA,2024,Q2,2024-Q2
USA,US,state,FL,2024,Q2,2024-Q2
USA,US,state,NJ,2024,Q2,2024-Q2
USA,US,state,NY,2024,Q2,2024-Q2


In [0]:
%sql
SELECT *
FROM default.usaspending_state_quarter_gold

country,country_code,geo_level,state,year,quarter,period,total_obligations,transaction_count
USA,US,state,NJ,2024,Q2,2024-Q2,7.87020199433E9,0
USA,US,state,NY,2024,Q2,2024-Q2,2.973427289062E10,0
USA,US,state,TX,2024,Q1,2024-Q1,2.570212140284E10,0
USA,US,state,PA,2024,Q1,2024-Q1,1.285273258249E10,0
USA,US,state,FL,2024,Q1,2024-Q1,1.7965954629370003E10,0
USA,US,state,NJ,2024,Q1,2024-Q1,7.42079132362E9,0
USA,US,state,NY,2024,Q1,2024-Q1,2.80219087863E10,0
USA,US,state,CA,2024,Q1,2024-Q1,5.266828891857E10,0
USA,US,state,PA,2024,Q2,2024-Q2,1.861540130384E10,0
USA,US,state,CA,2024,Q2,2024-Q2,5.621595714562E10,0
